In [0]:
# Gold Layer: Delivery SLA Report

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, sum as spark_sum,
    when, round as spark_round,
    datediff, current_timestamp, lit
)

spark = SparkSession.builder.getOrCreate()

GOLD_TABLE = "ecomstore.ecomlake.gold_delivery_sla_report"

# Read Silver Tables 
shipments = spark.read.table("ecomstore.ecomlake.silver_shipments")
orders = spark.read.table("ecomstore.ecomlake.silver_orders").select(
    "order_id", "shipping_address_city", "shipping_address_state", "channel", "status"
)
order_seller = (
    spark.read.table("ecomstore.ecomlake.silver_order_items")   # Safe Pre-Aggregation to prevent Fan-Out
    .groupBy("order_id")
    .agg({"seller_id": "min"})
    .withColumnRenamed("min(seller_id)", "seller_id")
)

sellers = (
    spark.read.table("ecomstore.ecomlake.silver_sellers_scd")
    .filter(col("is_current") == True)
    .select("seller_id", "seller_name", "seller_tier")
)

# Build the Enriched Shipment Base
enriched = (
    shipments.alias("s")
    .join(orders.alias("o"), on="order_id", how="left")
    .join(order_seller.alias("os"), on="order_id", how="left")
    .join(sellers.alias("sel"), on="seller_id", how="left")
    .filter(col("o.status") != "cancelled")
)

sla_metrics = (
    enriched
    .groupBy("s.carrier", "o.shipping_address_city", "o.shipping_address_state", "sel.seller_tier")
    .agg(
        count("s.shipment_id").alias("total_shipments"),
        count(when(col("s.shipment_status") == "delivered", True)).alias("delivered_count"),
        
        # Shipment must explicitly be 'delivered' before evaluating SLAs
        count(when(
            (col("s.shipment_status") == "delivered") & 
            (col("s.actual_delivery_date") <= col("s.promised_delivery_date")), True
        )).alias("on_time_count"),
        
        count(when(
            (col("s.shipment_status") == "delivered") & 
            (col("s.actual_delivery_date") > col("s.promised_delivery_date")), True
        )).alias("late_count"),
        
        spark_sum(datediff(col("s.actual_delivery_date"), col("s.dispatch_date"))).alias("sum_days_to_deliver")
    )
    .withColumnRenamed("shipping_address_city", "delivery_city")
    .withColumnRenamed("shipping_address_state", "delivery_state")
    .withColumn("_gold_processed_at", current_timestamp())
)

# Write to Gold Delta Table
(
    sla_metrics
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)